# 06 Start vLLM with LoRA Enabled

This notebook documents the local vLLM launch path. vLLM should run as an OpenAI-compatible server with LoRA enabled.

```mermaid
flowchart LR
    A[Qwen2.5-7B-Instruct] --> B[vLLM OpenAI server]
    C[adapters/*] --> B
    B --> D[/v1/chat/completions]
```

Dynamic adapter loading requires `VLLM_ALLOW_RUNTIME_LORA_UPDATING=True`.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "PROJECT_SPEC.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))

from llmops_demo.settings import settings, ensure_dirs

cfg = settings()
print(f"Project root: {PROJECT_ROOT}")
print(f"Base model: {cfg.base_model}")
print(f"Adapters: {cfg.adapters}")

## Compose command

Run this in a terminal from the project root. Expected output: vLLM logs ending with an OpenAI API server listening on port 8000.

In [ ]:
print("make serve")

## Equivalent vLLM command

Use this on an MLIS host or any Linux CUDA machine with vLLM installed.

In [ ]:
command = f'''
VLLM_ALLOW_RUNTIME_LORA_UPDATING=True \
python -m vllm.entrypoints.openai.api_server \
  --host ${'{'}VLLM_HOST:-0.0.0.0{'}'} \
  --port ${'{'}VLLM_PORT:-8000{'}'} \
  --model "{cfg.base_model}" \
  --served-model-name base \
  --enable-lora \
  --max-model-len {cfg.vllm_max_model_len if hasattr(cfg, "vllm_max_model_len") else 4096}
'''
print(command)

## Health check

Expected output: JSON model list containing `base` before adapters are loaded.

In [ ]:
import requests

response = requests.get(f"{cfg.vllm_base_url}/v1/models", headers={"Authorization": f"Bearer {cfg.vllm_api_key}"}, timeout=10)
print(response.status_code)
print(response.text[:1000])